In [ ]:
# KOMÓRKA 1: Importy
import time
import warnings
warnings.filterwarnings('ignore') # Wyciszenie ostrzeżeń (np. od sklearn czy xgboost) dla czystości wyników

import catboost
import lightgbm
import xgboost
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn import linear_model, svm, tree, ensemble, neighbors
from sklearn.metrics import (f1_score, roc_auc_score, classification_report, 
                             roc_curve, precision_recall_curve, auc)
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

# Importy z lokalnych modułów (wymaga folderu src/ z Twoimi plikami)
from src.outlier_generator import OutlierGenerator
from src.read_dataset import load, Dataset

# Ustawienia wizualne dla wykresów
sns.set_theme(style="whitegrid", palette="muted")

In [ ]:
# KOMÓRKA 2: Definicje algorytmów
tested_outlier_fns = [
    "uniform",
    "normal",
    "extreme"
]

# Modele binarne, które zostaną przekazane do wrappera
tested_binary_algorithms = [
    linear_model.LogisticRegression,
    svm.SVC,
    tree.DecisionTreeClassifier,
    ensemble.RandomForestClassifier,
    xgboost.XGBClassifier,
    catboost.CatBoostClassifier,
    lightgbm.LGBMClassifier
]

# Standardowe algorytmy do klasyfikacji jednoklasowej (baselines)
tested_oneclass_algorithms = [
    ensemble.IsolationForest,
    svm.OneClassSVM,
    neighbors.LocalOutlierFactor
]

# Zbiory danych do przetestowania (zgodnie z dokumentacją)
# Odkomentuj kolejne, jeśli je pobrałeś
datasets_to_test = [
    Dataset.ODDS, 
    # Dataset.HAR, 
    # Dataset.DFVE
]

In [ ]:
# KOMÓRKA 3: Funkcje do ewaluacji wizualnej
def evaluate_and_plot_curves(model, X_test, y_test, model_name, is_baseline=False):
    """Generuje raport klasyfikacji oraz rysuje krzywe ROC i PR."""
    preds = model.predict(X_test)
    
    # MAPOWANIE PREDYKCJI: ujednolicenie na standard zbioru testowego (0 = Inlier, 1 = Outlier)
    if is_baseline:
        # Algorytmy jednoklasowe sklearn: 1 (inlier), -1 (outlier)
        preds_mapped = np.where(preds == 1, 0, 1)
        # Pobieranie ciągłych wyników: wyższy wynik -> anomalia
        scores = -model.decision_function(X_test) if hasattr(model, "decision_function") else preds_mapped
    else:
        # OutlierGenerator: 1 (inlier), 0 (outlier)
        preds_mapped = np.where(preds == 1, 0, 1)
        try:
            # Prawdopodobieństwo klasy "0" z perspektywy modelu binarnego (czyli wygenerowanego outlier'a)
            scores = model.predict_proba(X_test)[:, 0]
        except AttributeError:
            if hasattr(model._estimator, "decision_function"):
                scores = -model._estimator.decision_function(X_test)
            else:
                scores = preds_mapped
                
    print(f"\n{'-'*55}")
    print(f"Raport Klasyfikacji: {model_name}")
    print(f"{'-'*55}")
    print(classification_report(y_test, preds_mapped, target_names=["Inlier (0)", "Outlier (1)"]))
    
    fpr, tpr, _ = roc_curve(y_test, scores, pos_label=1)
    roc_auc = auc(fpr, tpr)
    
    precision, recall, _ = precision_recall_curve(y_test, scores, pos_label=1)
    pr_auc = auc(recall, precision)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC AUC = {roc_auc:.3f}')
    ax1.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    ax1.set(xlim=[0.0, 1.0], ylim=[0.0, 1.05], xlabel='FPR', ylabel='TPR', title=f'ROC - {model_name}')
    ax1.legend(loc="lower right")
    
    ax2.plot(recall, precision, color='green', lw=2, label=f'PR AUC = {pr_auc:.3f}')
    ax2.set(xlim=[0.0, 1.0], ylim=[0.0, 1.05], xlabel='Recall', ylabel='Precision', title=f'PR - {model_name}')
    ax2.legend(loc="lower left")
    
    plt.tight_layout()
    plt.show()

def plot_pca_decision_boundary(model_class, model_kwargs, is_baseline, X_train, y_train, X_test, y_test, title):
    """Rzutuje dane na 2 wymiary za pomocą PCA i wizualizuje obszary decyzyjne."""
    pca = PCA(n_components=2, random_state=42)
    X_train_pca = pca.fit_transform(X_train)
    X_test_pca = pca.transform(X_test)
    
    # Trening TYLKO na inlierach
    X_train_normal_pca = X_train_pca[y_train == 0]
    
    model = model_class(**model_kwargs)
    model.fit(X_train_normal_pca)
    
    pad_x = (X_test_pca[:, 0].max() - X_test_pca[:, 0].min()) * 0.1
    pad_y = (X_test_pca[:, 1].max() - X_test_pca[:, 1].min()) * 0.1
    x_min, x_max = X_test_pca[:, 0].min() - pad_x, X_test_pca[:, 0].max() + pad_x
    y_min, y_max = X_test_pca[:, 1].min() - pad_y, X_test_pca[:, 1].max() + pad_y
    
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    grid_points = np.c_[xx.ravel(), yy.ravel()]
    
    Z = model.predict(grid_points)
    
    # Mapowanie siatki na standard 0/1
    if is_baseline:
        Z_mapped = np.where(Z == 1, 0, 1)
    else:
        Z_mapped = np.where(Z == 1, 0, 1)
        
    Z_mapped = Z_mapped.reshape(xx.shape)
    
    plt.figure(figsize=(10, 7))
    plt.contourf(xx, yy, Z_mapped, alpha=0.3, cmap='coolwarm')
    scatter = plt.scatter(X_test_pca[:, 0], X_test_pca[:, 1], c=y_test, cmap='coolwarm', edgecolor='k', alpha=0.8)
    
    handles, _ = scatter.legend_elements()
    plt.legend(handles, ["Inlier (Klasa 0)", "Outlier (Klasa 1)"], loc="upper right")
    plt.title(f"PCA Obszary Decyzyjne (2D) - {title}", fontsize=14)
    plt.xlabel(f"PCA 1 ({pca.explained_variance_ratio_[0]:.1%} wariancji)")
    plt.ylabel(f"PCA 2 ({pca.explained_variance_ratio_[1]:.1%} wariancji)")
    plt.tight_layout()
    plt.show()

In [ ]:
# KOMÓRKA 4: Pętla ucząca modele i zbierająca wyniki do tabeli
results = []

for dataset_name in datasets_to_test:
    print(f"\n{'='*50}\nPrzetwarzanie zbioru: {dataset_name}\n{'='*50}")
    
    # 1. Wczytanie i podział danych
    df, metadata = load(dataset_name)
    
    # Jeśli w DFVE masz dodatkową kolumnę 'label_ratio', warto ją wyrzucić ze zbioru cech
    cols_to_drop = ["label", "label_ratio", "activity"]
    X = df.drop(columns=[col for col in cols_to_drop if col in df.columns]).values
    y = df["label"].values
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    # POPRAWKA: Uczymy wyłącznie na przykładach "normalnych" (Inliers = 0)
    X_train_normal = X_train[y_train == 0]
    
    # 2. Testowanie klasyfikatorów binarnych + OutlierGenerator
    for clf_class in tested_binary_algorithms:
        for out_fn in tested_outlier_fns:
            model_name = clf_class.__name__
            experiment_name = f"{model_name} ({out_fn})"
            print(f"Trenowanie: {experiment_name}...")
            
            # Wyciszanie logów dla CatBoost i LightGBM
            kwargs = {'random_state': 42}
            if model_name == 'CatBoostClassifier': kwargs['verbose'] = False
            elif model_name == 'LGBMClassifier': kwargs['verbose'] = -1
            elif model_name == 'XGBClassifier': kwargs['eval_metric'] = 'logloss'
            
            base_estimator = clf_class(**kwargs)
            model = OutlierGenerator(estimator=base_estimator, outliers_fn=out_fn, random_state=42)
            
            start_time = time.time()
            model.fit(X_train_normal) 
            preds = model.predict(X_test)
            fit_time = time.time() - start_time
            
            # Mapowanie i ewaluacja
            preds_mapped = np.where(preds == 1, 0, 1)
            f1 = f1_score(y_test, preds_mapped, pos_label=1)
            roc = roc_auc_score(y_test, preds_mapped) # Aproksymacja ROC na twardych etykietach dla szybkiej tabeli
            
            results.append({
                "Dataset": dataset_name,
                "Model_Type": "Binary + Generator",
                "Algorithm": model_name,
                "Method": out_fn,
                "Full_Name": experiment_name,
                "F1_Score": f1,
                "ROC_AUC": roc,
                "Time_s": fit_time
            })
            
    # 3. Testowanie modeli Baseline (Jednoklasowych)
    print("\n--- Testowanie algorytmów jednoklasowych (Baseline) ---")
    for clf_class in tested_oneclass_algorithms:
        model_name = clf_class.__name__
        print(f"Trenowanie baseline: {model_name}...")
        
        # Specjalna obsługa dla LOF
        kwargs = {}
        if model_name == 'LocalOutlierFactor': kwargs['novelty'] = True
        elif model_name == 'IsolationForest': kwargs['random_state'] = 42
            
        model = clf_class(**kwargs)
            
        start_time = time.time()
        model.fit(X_train_normal) 
        preds = model.predict(X_test)
        fit_time = time.time() - start_time
        
        # Mapowanie i ewaluacja (-1/1 -> 1/0)
        preds_mapped = np.where(preds == 1, 0, 1)
        f1 = f1_score(y_test, preds_mapped, pos_label=1)
        roc = roc_auc_score(y_test, preds_mapped)
        
        results.append({
            "Dataset": dataset_name,
            "Model_Type": "One-Class Baseline",
            "Algorithm": model_name,
            "Method": "baseline",
            "Full_Name": model_name,
            "F1_Score": f1,
            "ROC_AUC": roc,
            "Time_s": fit_time
        })

print("\nGotowe! Eksperymenty zakończone pomyślnie.")

In [ ]:
# KOMÓRKA 5: Analiza i wizualizacja w zbiorczej tabeli
df_results = pd.DataFrame(results)
display(df_results.sort_values(by="F1_Score", ascending=False))

def plot_results(df, metric="F1_Score"):
    g = sns.catplot(
        data=df, 
        kind="bar",
        x="Full_Name", 
        y=metric, 
        col="Dataset",
        hue="Model_Type", 
        col_wrap=1,      
        height=6, 
        aspect=2.5,      
        dodge=False
    )
    
    g.set_axis_labels("Algorytm / Metoda", metric)
    g.set_titles("Wydajność modeli: {col_name}")
    
    for ax in g.axes.flat:
        for label in ax.get_xticklabels():
            label.set_rotation(45)
            label.set_horizontalalignment('right')
            
    plt.tight_layout()
    plt.show()

print("\n--- Wykres zbiorczy F1-Score ---")
plot_results(df_results, metric="F1_Score")
print("\n--- Wykres zbiorczy ROC AUC ---")
plot_results(df_results, metric="ROC_AUC")

In [ ]:
# KOMÓRKA 6: Pogłębiona analiza na wybranym zbiorze danych
print("\nGenerowanie analiz pogłębionych (PCA i Krzywe) dla zbioru ODDS...")

# Pobieramy raz jeszcze do izolacji dla celów wizualizacyjnych
df_eval, _ = load(Dataset.ODDS)
X_eval = df_eval.drop(columns=["label"]).values
y_eval = df_eval["label"].values

X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X_eval, y_eval, test_size=0.2, random_state=42, stratify=y_eval
)

# Do trenowania PCA bierzemy tylko INLIERS
X_train_normal_e = X_train_e[y_train_e == 0]

# --- 1. Analiza: Przykładowy model Generatora (np. Random Forest + Normal) ---
rf_base = ensemble.RandomForestClassifier(random_state=42)
model_rf_gen = OutlierGenerator(estimator=rf_base, outliers_fn="normal", random_state=42)
model_rf_gen.fit(X_train_normal_e)

evaluate_and_plot_curves(
    model=model_rf_gen, 
    X_test=X_test_e, 
    y_test=y_test_e, 
    model_name="Random Forest + Normal Outlier Generator", 
    is_baseline=False
)

plot_pca_decision_boundary(
    model_class=OutlierGenerator,
    model_kwargs={"estimator": ensemble.RandomForestClassifier(random_state=42), "outliers_fn": "normal", "random_state": 42},
    is_baseline=False,
    X_train=X_train_e, y_train=y_train_e, X_test=X_test_e, y_test=y_test_e,
    title="Random Forest (Generowane Anomalie)"
)

# --- 2. Analiza: Baseline (Isolation Forest) ---
model_iso = ensemble.IsolationForest(random_state=42)
model_iso.fit(X_train_normal_e)

evaluate_and_plot_curves(
    model=model_iso, 
    X_test=X_test_e, 
    y_test=y_test_e, 
    model_name="Isolation Forest (Baseline)", 
    is_baseline=True
)

plot_pca_decision_boundary(
    model_class=ensemble.IsolationForest,
    model_kwargs={"random_state": 42},
    is_baseline=True,
    X_train=X_train_e, y_train=y_train_e, X_test=X_test_e, y_test=y_test_e,
    title="Isolation Forest (Klasyczny Model)"
)